# **Notebook 2: Text Encoder Selection for Dual-View X-ray Fusion**

**Description:** This notebook compares candidate clinical text encoders for the multimodal chest X-ray classifier while keeping the image branch, saved splits, label space, and training protocol aligned with Notebook 01. It evaluates CheXbert, Bio_ClinicalBERT, and RadBERT in both text-only and image/text fusion settings.

**Main Questions This Notebook Should Answer:**
- Does a domain-specific text encoder improve classification beyond the dual-view image baseline?
- Does the improvement show up in accuracy, macro F1, or both?
- Which text encoder should be carried forward into the residual gated fusion experiment?

---

# **Part 1: Configuration**
### **Imports**


In [ ]:
from pathlib import Path
import sys
import gc
import json
import math
import os
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder, label_binarize

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from transformers import AutoModel, AutoTokenizer, BertConfig, BertModel
from transformers.utils.hub import cached_file

# Ensure Python can find the local xray_fusion package.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "src" / "xray_fusion").exists()),
    Path.cwd(),
)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from xray_fusion.config import DEFAULT_IMAGE_SIZE, DEFAULT_MAX_LEN, find_project_root
from xray_fusion import data as xray_data
from xray_fusion import models as xray_models
from xray_fusion import training as xray_training
from xray_fusion import evaluation as xray_eval
from xray_fusion import segmentation as xray_segmentation
from xray_fusion import gradcam as xray_gradcam

BASE_SEED = 42
EXPERIMENT_SEEDS = [42]

seed_everything = xray_training.seed_everything
seed_everything(BASE_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


### **Paths**


In [ ]:
project_root = Path.cwd().resolve()
if not (project_root / "splits").exists():
    project_root = project_root.parent

base_dir = project_root
split_dir = base_dir / "splits"

train_csv = split_dir / "train.csv"
val_csv = split_dir / "val.csv"
test_csv = split_dir / "test.csv"

### **Configurations**


In [ ]:
HF_CACHE_DIR = base_dir / "hf_cache"
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 10

LR_TEXT = 2e-5
LR_IMAGE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_LEN = 64
FREEZE_BERT = True
FREEZE_IMAGE_BACKBONE = False

LR_SCHEDULE_NAME = "constant"
EXPERIMENT_NAME = "experiment_02_text_encoder_benchmark"


### **Benchmark Setup**


In [ ]:
OUTPUT_DIR = base_dir / "Experiments" / "outputs" / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESOLVED_MODEL_CACHE_DIR = OUTPUT_DIR / "resolved_hf_models"
RESOLVED_MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Saving artifacts to:", OUTPUT_DIR)

TEXT_ENCODER_SPECS = [
    {
        "name": "Bio_ClinicalBERT",
        "slug": "bio_clinicalbert",
        "loader": "auto",
        "model_name": "emilyalsentzer/Bio_ClinicalBERT",
        "tokenizer_name": "emilyalsentzer/Bio_ClinicalBERT",
    },
    {
        "name": "CheXbert",
        "slug": "chexbert",
        "loader": "chexbert",
        "repo_id": "StanfordAIMI/RRG_scorers",
        "filename": "chexbert.pth",
        "tokenizer_name": "bert-base-uncased",
    },
    {
        "name": "RadBERT",
        "slug": "radbert",
        "loader": "auto",
        "model_name": "StanfordAIMI/RadBERT",
        "tokenizer_name": "StanfordAIMI/RadBERT",
    },
]

print("Experiment seeds:", EXPERIMENT_SEEDS)
print("Text encoders configured:")
for spec in TEXT_ENCODER_SPECS:
    print(f" - {spec['name']} | tokenizer: {spec['tokenizer_name']}")


### **Load precomputed train, validation, and test splits**


In [ ]:
train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())


---

# **Part 2: Dataset Preparation**

The text-only, image-only, and fusion experiments all reuse the same labels, image transforms, split files, and class weighting so the encoder comparison stays controlled.
### **Encode Labels**


In [ ]:
label_encoder = LabelEncoder()
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label_enc"] = label_encoder.fit_transform(train_df["clean_label"])
val_df["label_enc"] = label_encoder.transform(val_df["clean_label"])
test_df["label_enc"] = label_encoder.transform(test_df["clean_label"])

label_names = list(label_encoder.classes_)
num_classes = len(label_names)

print("Classes:", label_names)

### **Image Transforms**


In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


### **Dataset Classes**


In [ ]:
class ClinicalTextDataset(xray_data.XRayTextDataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        super().__init__(df, tokenizer, max_len=max_len, label_key="labels")

class XRayMultimodalDataset(xray_data.XRayMultimodalDataset):
    def __init__(self, df, tokenizer, image_transform=None, max_len=MAX_LEN):
        super().__init__(df, tokenizer=tokenizer, image_transform=image_transform, max_len=max_len, label_key="labels")

class XRayDualImageDataset(xray_data.XRayDualImageDictDataset):
    def __init__(self, df, image_transform=None):
        super().__init__(df, image_transform=image_transform, label_key="labels")


### **Build dataloader helpers**


In [ ]:
def make_torch_generator(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def make_loader(dataset, seed, shuffle):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        generator=make_torch_generator(seed) if shuffle else None,
    )


def load_tokenizer_for_encoder(encoder_spec):
    tokenizer_name = encoder_spec["tokenizer_name"]
    local_source = resolve_tokenizer_source(tokenizer_name)
    tokenizer_source = local_source if local_source is not None else tokenizer_name
    local_only = local_source is not None
    print(f"Loading tokenizer for {encoder_spec['name']}: {tokenizer_source}")
    return AutoTokenizer.from_pretrained(
        tokenizer_source,
        cache_dir=HF_CACHE_DIR,
        local_files_only=local_only,
    )


def build_text_loaders(tokenizer, seed):
    train_ds = ClinicalTextDataset(train_df, tokenizer)
    val_ds = ClinicalTextDataset(val_df, tokenizer)
    test_ds = ClinicalTextDataset(test_df, tokenizer)
    return (
        make_loader(train_ds, seed=seed, shuffle=True),
        make_loader(val_ds, seed=seed, shuffle=False),
        make_loader(test_ds, seed=seed, shuffle=False),
    )


def build_multimodal_loaders(tokenizer, seed):
    train_ds = XRayMultimodalDataset(train_df, tokenizer, image_transform=image_transform)
    val_ds = XRayMultimodalDataset(val_df, tokenizer, image_transform=image_transform)
    test_ds = XRayMultimodalDataset(test_df, tokenizer, image_transform=image_transform)
    return (
        make_loader(train_ds, seed=seed, shuffle=True),
        make_loader(val_ds, seed=seed, shuffle=False),
        make_loader(test_ds, seed=seed, shuffle=False),
    )


def build_image_loaders(seed):
    train_ds = XRayDualImageDataset(train_df, image_transform=image_transform)
    val_ds = XRayDualImageDataset(val_df, image_transform=image_transform)
    test_ds = XRayDualImageDataset(test_df, image_transform=image_transform)
    return (
        make_loader(train_ds, seed=seed, shuffle=True),
        make_loader(val_ds, seed=seed, shuffle=False),
        make_loader(test_ds, seed=seed, shuffle=False),
    )

sample_img_loader, _, _ = build_image_loaders(BASE_SEED)
sample_batch = next(iter(sample_img_loader))
for key, value in sample_batch.items():
    print(key, tuple(value.shape))


### **Class Weights**


In [ ]:
class_counts = train_df["clean_label"].value_counts()
class_weights = torch.tensor(
    [len(train_df) / class_counts[label] for label in label_names],
    dtype=torch.float32,
    device=device,
)


---

# **Part 3: Model and Training Utilities**

### **Model Classes**


In [ ]:
get_cls_embedding = xray_models.get_cls_embedding
hf_repo_cache_dir = lambda repo_id, cache_dir=HF_CACHE_DIR: xray_models.hf_repo_cache_dir(repo_id, cache_dir)
iter_hf_snapshots = lambda repo_id, cache_dir=HF_CACHE_DIR: xray_models.iter_hf_snapshots(repo_id, cache_dir)
first_cached_file = lambda repo_id, filenames, cache_dir=HF_CACHE_DIR: xray_models.first_cached_file(repo_id, filenames, cache_dir)
copy_if_needed = xray_models.copy_if_needed
materialize_cached_repo = lambda repo_id, required_files, optional_files=None, cache_dir=HF_CACHE_DIR: xray_models.materialize_cached_repo(
    repo_id, required_files, optional_files=optional_files, cache_dir=cache_dir, resolved_dir=RESOLVED_MODEL_CACHE_DIR
)
resolve_model_source = lambda model_name: xray_models.resolve_model_source(model_name, HF_CACHE_DIR, RESOLVED_MODEL_CACHE_DIR)
resolve_tokenizer_source = lambda tokenizer_name: xray_models.resolve_tokenizer_source(tokenizer_name, HF_CACHE_DIR, RESOLVED_MODEL_CACHE_DIR)
load_auto_backbone = lambda model_name, cache_dir=None: xray_models.load_auto_backbone(
    model_name, cache_dir=cache_dir or HF_CACHE_DIR, resolved_dir=RESOLVED_MODEL_CACHE_DIR
)
load_chexbert_backbone = xray_models.load_chexbert_backbone

def load_text_backbone(spec, cache_dir=None):
    if spec["loader"] == "auto":
        backbone = load_auto_backbone(spec["model_name"], cache_dir=cache_dir)
    else:
        backbone = load_chexbert_backbone(
            spec["repo_id"],
            spec["filename"],
            cache_dir=cache_dir
        )

    return backbone, backbone.config.hidden_size

class TextOnlyClassifier(xray_models.TextOnlyClassifier):
    def __init__(self, encoder_spec, num_classes, freeze_bert=False, cache_dir=None):
        super().__init__(
            encoder_spec,
            num_classes,
            freeze_bert=freeze_bert,
            cache_dir=cache_dir,
            resolved_dir=RESOLVED_MODEL_CACHE_DIR
        )

DualImageEncoder = xray_models.DualImageEncoder
DualImageClassifier = xray_models.DualImageClassifier

class FusionClassifier(xray_models.FusionClassifier):
    def __init__(
        self,
        encoder_spec,
        num_classes,
        freeze_bert=False,
        freeze_image_backbone=False,
        cache_dir=None
    ):
        super().__init__(
            encoder_spec,
            num_classes,
            freeze_bert=freeze_bert,
            freeze_image_backbone=freeze_image_backbone,
            cache_dir=cache_dir,
            resolved_dir=RESOLVED_MODEL_CACHE_DIR,
        )

### **Training Functions**


In [ ]:
configure_optimizer = lambda model, lr_text=LR_TEXT, lr_image=LR_IMAGE, weight_decay=WEIGHT_DECAY: xray_training.configure_optimizer(
    model, lr_text=lr_text, lr_image=lr_image, weight_decay=weight_decay
)
build_scheduler = xray_training.build_scheduler
move_batch_to_device = xray_training.move_task_batch_to_device
forward_task = xray_training.forward_task
train_one_epoch = xray_training.train_one_epoch
evaluate_model = xray_training.evaluate_model


### **Metric and Artifact Helpers**


In [ ]:
safe_metric = xray_eval.safe_metric
compute_probability_metrics = xray_eval.compute_probability_metrics
compute_metrics = xray_eval.compute_metrics
save_json = xray_eval.save_json
plot_history = xray_eval.plot_history
classification_report_dict = xray_eval.classification_report_dict

def save_confusion_matrix_figure(cm, title, save_path):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks(range(len(label_names)))
    ax.set_yticks(range(len(label_names)))
    ax.set_xticklabels(label_names, rotation=45, ha="right")
    ax.set_yticklabels(label_names)
    for i in range(len(label_names)):
        for j in range(len(label_names)):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def checkpoint_payload(model, optimizer, scheduler, encoder_spec, task_type, seed, best_epoch, best_val_acc, config_extra=None):
    return {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "encoder_spec": encoder_spec,
        "task_type": task_type,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "config": config_extra or {},
    }

def save_split_artifacts(split_name, metrics, report, cm, run_dir, title_prefix):
    save_json(metrics, run_dir / f"{split_name}_metrics.json")
    save_json(report, run_dir / f"{split_name}_classification_report.json")
    pd.DataFrame(cm, index=label_names, columns=label_names).to_csv(run_dir / f"{split_name}_confusion_matrix.csv")
    save_confusion_matrix_figure(cm, f"{title_prefix} {split_name.title()} Confusion Matrix", run_dir / f"{split_name}_confusion_matrix.png")

def extract_per_class_rows(report, encoder_name, encoder_slug, task_type, seed, split_name):
    rows = []
    for class_name in label_names:
        class_metrics = report.get(class_name, {})
        rows.append({
            "encoder_name": encoder_name,
            "encoder_slug": encoder_slug,
            "task_type": task_type,
            "seed": seed,
            "split": split_name,
            "class_name": class_name,
            "precision": class_metrics.get("precision", np.nan),
            "recall": class_metrics.get("recall", np.nan),
            "f1_score": class_metrics.get("f1-score", np.nan),
            "support": class_metrics.get("support", np.nan),
        })
    return pd.DataFrame(rows)


### **Run Helper**


In [ ]:
def train_and_evaluate_run(encoder_spec, task_type, seed):
    seed_everything(seed)

    if task_type == "image_only":
        encoder_name = "DualImageResNet18"
        encoder_slug = "dual_image_resnet18"
        run_slug = "dual_image_baseline"
        tokenizer_name = None
    else:
        encoder_name = encoder_spec["name"]
        encoder_slug = encoder_spec["slug"]
        run_slug = f"{encoder_slug}_{task_type}"
        tokenizer_name = encoder_spec["tokenizer_name"]

    run_dir = OUTPUT_DIR / run_slug / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "best_checkpoint.pt"

    if task_type == "image_only":
        model = DualImageClassifier(
            num_classes=num_classes,
            freeze_image_backbone=FREEZE_IMAGE_BACKBONE,
        ).to(device)
        train_loader, val_loader, test_loader = build_image_loaders(seed)
        run_title = "Dual-Image ResNet18 Baseline"
    elif task_type == "text_only":
        tokenizer = load_tokenizer_for_encoder(encoder_spec)
        model = TextOnlyClassifier(
            encoder_spec=encoder_spec,
            num_classes=num_classes,
            freeze_bert=FREEZE_BERT,
            cache_dir=HF_CACHE_DIR,
        ).to(device)
        train_loader, val_loader, test_loader = build_text_loaders(tokenizer, seed)
        run_title = f"{encoder_name} Text-Only"
    elif task_type == "fusion":
        tokenizer = load_tokenizer_for_encoder(encoder_spec)
        model = FusionClassifier(
            encoder_spec=encoder_spec,
            num_classes=num_classes,
            freeze_bert=FREEZE_BERT,
            freeze_image_backbone=FREEZE_IMAGE_BACKBONE,
            cache_dir=HF_CACHE_DIR,
        ).to(device)
        train_loader, val_loader, test_loader = build_multimodal_loaders(tokenizer, seed)
        run_title = f"{encoder_name} Fusion"
    else:
        raise ValueError(f"Unsupported task_type: {task_type}")

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = configure_optimizer(model)
    scheduler = build_scheduler(optimizer)

    history = []
    best_val_acc = -math.inf
    best_epoch = -1

    for epoch in range(1, EPOCHS + 1):
        print(f"{run_title} - Epoch {epoch}/{EPOCHS}")

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            task_type=task_type,
            desc=f"Train {run_slug}",
        )

        val_loss, val_acc, _, _, _ = evaluate_model(
            model,
            val_loader,
            criterion,
            device,
            task_type=task_type,
            desc=f"Val {run_slug}",
        )

        scheduler.step()

        lr_values = [group["lr"] for group in optimizer.param_groups]
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "lr_min": min(lr_values),
                "lr_max": max(lr_values),
            }
        )

        print(
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f}"
        )
        print()

        if val_acc >= best_val_acc:
            best_val_acc = float(val_acc)
            best_epoch = epoch
            torch.save(
                checkpoint_payload(
                    model,
                    optimizer,
                    scheduler,
                    encoder_spec,
                    task_type,
                    seed,
                    epoch,
                    best_val_acc,
                ),
                checkpoint_path,
            )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "training_history.csv", index=False)
    plot_history(history_df, run_title, run_dir / "training_curves.png")

    best_checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(best_checkpoint["model_state_dict"])

    val_loss, val_acc, val_y_true, val_y_pred, val_y_prob = evaluate_model(
        model,
        val_loader,
        criterion,
        device,
        task_type=task_type,
        desc=f"Best Val {run_slug}",
    )
    test_loss, test_acc, test_y_true, test_y_pred, test_y_prob = evaluate_model(
        model,
        test_loader,
        criterion,
        device,
        task_type=task_type,
        desc=f"Test {run_slug}",
    )

    val_metrics = compute_metrics(val_y_true, val_y_pred, val_y_prob, label_names)
    test_metrics = compute_metrics(test_y_true, test_y_pred, test_y_prob, label_names)
    val_report = classification_report_dict(val_y_true, val_y_pred, label_names)
    test_report = classification_report_dict(test_y_true, test_y_pred, label_names)
    val_cm = confusion_matrix(val_y_true, val_y_pred, labels=np.arange(len(label_names)))
    test_cm = confusion_matrix(test_y_true, test_y_pred, labels=np.arange(len(label_names)))

    save_split_artifacts("validation", val_metrics, val_report, val_cm, run_dir, run_title)
    save_split_artifacts("test", test_metrics, test_report, test_cm, run_dir, run_title)

    run_summary = {
        "encoder_name": encoder_name,
        "encoder_slug": encoder_slug,
        "task_type": task_type,
        "seed": seed,
        "tokenizer_name": tokenizer_name,
        "best_epoch": best_epoch,
        "val_loss": float(val_loss),
        "val_accuracy": float(val_acc),
        "val_macro_f1": float(val_metrics["macro_f1"]),
        "val_macro_auroc": float(val_metrics["macro_auroc"]),
        "val_macro_auprc": float(val_metrics["macro_auprc"]),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_metrics["macro_f1"]),
        "test_macro_auroc": float(test_metrics["macro_auroc"]),
        "test_macro_auprc": float(test_metrics["macro_auprc"]),
        "checkpoint_path": str(checkpoint_path),
        "run_dir": str(run_dir),
    }
    save_json(run_summary, run_dir / "run_summary.json")

    per_class_frames = [
        extract_per_class_rows(val_report, encoder_name, encoder_slug, task_type, seed, "validation"),
        extract_per_class_rows(test_report, encoder_name, encoder_slug, task_type, seed, "test"),
    ]
    run_per_class_df = pd.concat(per_class_frames, ignore_index=True)
    run_per_class_df.to_csv(run_dir / "per_class_metrics_all_splits.csv", index=False)

    display(history_df.tail(1))
    display(run_per_class_df)

    print(f"{run_title} validation accuracy:", round(run_summary["val_accuracy"], 4))
    print(f"{run_title} test accuracy:", round(run_summary["test_accuracy"], 4))
    print(f"{run_title} test macro F1:", round(run_summary["test_macro_f1"], 4))
    print(f"{run_title} test macro AUROC:", round(run_summary["test_macro_auroc"], 4))
    print(f"{run_title} test macro AUPRC:", round(run_summary["test_macro_auprc"], 4))
    print("Best checkpoint:", checkpoint_path)
    print()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return run_summary, run_per_class_df


### **Benchmark Helpers**


In [ ]:
def get_encoder_spec(name_or_slug):
    requested = str(name_or_slug).lower()
    for spec in TEXT_ENCODER_SPECS:
        candidates = {spec["name"].lower(), spec["slug"].lower()}
        if requested in candidates:
            return spec
    available = ", ".join(spec["name"] for spec in TEXT_ENCODER_SPECS)
    raise ValueError(f"Unknown encoder '{name_or_slug}'. Available encoders: {available}")


def clean_imported_summary(summary, source_label):
    summary = dict(summary)
    for metadata_key in ("tokenizer" + "_note", "description"):
        summary.pop(metadata_key, None)
    summary["tokenizer_name"] = summary.get("tokenizer_name") or "not_applicable"
    summary["source_notebook"] = source_label
    return summary


def load_saved_dual_image_baseline():
    baseline_dir = OUTPUT_DIR / "dual_image_baseline"
    rows = []
    per_class_tables = []

    for seed in EXPERIMENT_SEEDS:
        run_dir = baseline_dir / f"seed_{seed}"
        summary_path = run_dir / "run_summary.json"
        per_class_path = run_dir / "per_class_metrics_all_splits.csv"
        if not summary_path.exists() or not per_class_path.exists():
            return [], []

        with open(summary_path, "r", encoding="utf-8") as handle:
            rows.append(clean_imported_summary(json.load(handle), "02 saved dual-image baseline artifacts"))
        per_class_tables.append(pd.read_csv(per_class_path))

    return rows, per_class_tables


def import_saved_image_baseline(run_summaries, per_class_tables):
    baseline_rows, baseline_per_class_tables = load_saved_dual_image_baseline()
    if not baseline_rows:
        raise FileNotFoundError(
            "Saved dual-image baseline artifacts were not found. Run Notebook 01/previous baseline first, "
            "or restore Experiments/outputs/experiment_02_text_encoder_benchmark/dual_image_baseline/seed_* artifacts."
        )

    run_summaries.extend(baseline_rows)
    per_class_tables.extend(baseline_per_class_tables)

    baseline_df = pd.DataFrame(baseline_rows).sort_values("seed").reset_index(drop=True)
    baseline_per_class_df = pd.concat(baseline_per_class_tables, ignore_index=True)

    display(baseline_df)
    display(baseline_per_class_df.head(10))
    print("Imported saved dual-image baseline artifact for seed 42.")
    print("Seed 42 test accuracy:", round(float(baseline_df["test_accuracy"].iloc[0]), 4))
    print("Seed 42 test macro F1:", round(float(baseline_df["test_macro_f1"].iloc[0]), 4))
    print()

    return baseline_df, baseline_per_class_df


def train_encoder_variation(encoder_spec, run_summaries, per_class_tables):
    print("=" * 80)
    print(f"Starting runs for {encoder_spec['name']}")
    print("=" * 80)

    for task_type in ["text_only", "fusion"]:
        for seed in EXPERIMENT_SEEDS:
            run_summary, run_per_class_df = train_and_evaluate_run(encoder_spec, task_type, seed)
            run_summaries.append(run_summary)
            per_class_tables.append(run_per_class_df)

def build_benchmark_summary_tables(run_summaries, per_class_tables):
    run_summary_df = pd.DataFrame(run_summaries)
    if not run_summary_df.empty:
        task_order = {"image_only": 0, "text_only": 1, "fusion": 2}
        run_summary_df["task_rank"] = run_summary_df["task_type"].map(task_order).fillna(99)
        run_summary_df = run_summary_df.sort_values(
            ["task_rank", "encoder_name", "seed"],
            ascending=[True, True, True],
        ).drop(columns=["task_rank"]).reset_index(drop=True)
        run_summary_df.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)

        aggregate_rows = []
        group_cols = ["encoder_name", "encoder_slug", "task_type", "tokenizer_name"]
        metric_cols = ["test_accuracy", "test_macro_f1", "test_macro_auroc", "test_macro_auprc"]
        for group_key, group_df in run_summary_df.groupby(group_cols, dropna=False):
            row = dict(zip(group_cols, group_key))
            for metric_col in metric_cols:
                metric_values = group_df[metric_col].dropna().astype(float)
                row[f"{metric_col}_mean"] = float(metric_values.mean()) if not metric_values.empty else np.nan
            aggregate_rows.append(row)

        aggregate_summary_df = pd.DataFrame(aggregate_rows)
        aggregate_summary_df = aggregate_summary_df.sort_values(
            ["task_type", "test_accuracy_mean", "test_macro_f1_mean"],
            ascending=[True, False, False],
        ).reset_index(drop=True)
        aggregate_summary_df.to_csv(OUTPUT_DIR / "aggregate_summary.csv", index=False)
        save_json(aggregate_summary_df.to_dict(orient="records"), OUTPUT_DIR / "aggregate_summary.json")
    else:
        aggregate_summary_df = pd.DataFrame()

    if per_class_tables:
        per_class_summary_df = pd.concat(per_class_tables, ignore_index=True)
        per_class_summary_df.to_csv(OUTPUT_DIR / "per_class_summary.csv", index=False)
    else:
        per_class_summary_df = pd.DataFrame()

    print("Benchmark summaries refreshed.")
    if not run_summary_df.empty:
        print("Run summary CSV:", OUTPUT_DIR / "run_summary.csv")
    if not aggregate_summary_df.empty:
        print("Aggregate summary CSV:", OUTPUT_DIR / "aggregate_summary.csv")
    if not per_class_summary_df.empty:
        print("Per-class summary CSV:", OUTPUT_DIR / "per_class_summary.csv")
    return run_summary_df, aggregate_summary_df, per_class_summary_df


### **Initialize Benchmark Tracking**


In [ ]:
run_summaries = []
per_class_tables = []


---

# **Part 4: Train Models**

### **Import Saved Dual-Image Baseline**


In [ ]:
import_saved_image_baseline(run_summaries, per_class_tables)


### **Train CheXbert Text Encoder Variation**


In [ ]:
chexbert_spec = get_encoder_spec("chexbert")
train_encoder_variation(chexbert_spec, run_summaries, per_class_tables)


### **Train Bio_ClinicalBERT Text Encoder Variation**


In [ ]:
clinicalbert_spec = get_encoder_spec("bio_clinicalbert")
train_encoder_variation(clinicalbert_spec, run_summaries, per_class_tables)


### **Train RadBERT Text Encoder Variation**


In [ ]:
radbert_spec = get_encoder_spec("radbert")
train_encoder_variation(radbert_spec, run_summaries, per_class_tables)


---

# **Part 5: Results and Reporting**

### **Export Benchmark Summaries**


In [ ]:
run_summary_df, aggregate_summary_df, per_class_summary_df = build_benchmark_summary_tables(
    run_summaries,
    per_class_tables,
)


### **Review the Exported Summaries**


In [ ]:
def hide_seed_columns(df):
    return df.drop(columns=["seed", "n_seeds", "seeds"], errors="ignore")

if 'run_summary_df' in globals() and not run_summary_df.empty:
    display(hide_seed_columns(run_summary_df))

if 'aggregate_summary_df' in globals() and not aggregate_summary_df.empty:
    display(hide_seed_columns(aggregate_summary_df))

if 'per_class_summary_df' in globals() and not per_class_summary_df.empty:
    display(hide_seed_columns(per_class_summary_df).head(12))


### **Benchmark Result Interpretation**

The encoder comparison is a narrowing step, not the final model decision. Bio_ClinicalBERT is the best handoff choice because it gives the strongest fusion accuracy and a stable clinical-language backbone for the gated-fusion test, while CheXbert's stronger macro F1 remains an important minority-class caveat. The main takeaway is that image-based fusion is still carrying performance, and later notebooks need to test whether architecture and preprocessing changes improve balanced abnormal-class behavior.

---

# **Part 6: Conclusion**

The fusion results did not produce a single winner across every metric. Bio_ClinicalBERT fusion had the strongest accuracy among the text-encoder fusion runs at 0.756, while CheXbert fusion had the strongest macro F1 at 0.415. Text-only models stayed much weaker than image-based models, which confirms that the report text alone is not enough to solve the task under this five-class label setup.

Bio_ClinicalBERT was chosen as the winner for the next stage because it gave the strongest overall held-out fusion accuracy while remaining a clinically appropriate, stable encoder for the project pipeline. That mattered because Notebook 03 was designed to test the fusion architecture itself, not to continue swapping language models. Carrying forward the most accurate Bio_ClinicalBERT fusion run gave the gated-fusion ablation a clear and recoverable backbone while keeping the next experiment focused on whether residual gated fusion improves the selected multimodal model.

It is also worth mentioning that CheXbert's stronger macro F1 is still an important caveat. It suggests CheXbert handled the minority-class balance better in this comparison, so Bio_ClinicalBERT should not be described as dominating every metric. Bio_ClinicalBERT wins the handoff because it combines the best fusion accuracy, strong clinical-language relevance, and a clean path into the residual gated fusion architecture that later notebooks evaluate.

---